# Stream A — 02 Visual Grounding

This notebook computes CLIP-based grounding scores for rows with local images.

It uses the new processed manifest layer and flat image folder.

For an 8 GB RTX 4060 Ti, default image batch size is conservative (`16`). Increase only if VRAM allows.


In [1]:
from pathlib import Path
import sys, pickle, time, math
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    STREAM_A_CACHE, ALL_MANIFEST, PROCESSED_IMAGES,
    FIGURES, TABLES
)

NER_CACHE = STREAM_A_CACHE / "ner"
VG_CACHE = STREAM_A_CACHE / "visual_grounding"
VG_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("VG_CACHE:", VG_CACHE)


ROOT: E:\eve-rag
VG_CACHE: E:\eve-rag\cache\stream_a\visual_grounding


In [2]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    torch.backends.cudnn.benchmark = True

# Conservative settings for 8 GB GPUs.
IMAGE_BATCH_SIZE = 16
TEXT_BATCH_SIZE = 128
USE_AMP = DEVICE == "cuda"
DEBUG_LIMIT_ROWS = None  # set to e.g. 500 for a quick test

print("IMAGE_BATCH_SIZE:", IMAGE_BATCH_SIZE)
print("TEXT_BATCH_SIZE:", TEXT_BATCH_SIZE)
print("USE_AMP:", USE_AMP)


Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM GB: 8.0
IMAGE_BATCH_SIZE: 16
TEXT_BATCH_SIZE: 128
USE_AMP: True


In [3]:
# Load OpenCLIP.
try:
    import open_clip
except Exception as e:
    raise RuntimeError(
        "open_clip is required. Install with: pip install open_clip_torch"
    ) from e

MODEL_NAME = "ViT-B-32"
PRETRAINED = "openai"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
clip_tokenizer = open_clip.get_tokenizer(MODEL_NAME)
clip_model = clip_model.to(DEVICE).eval()

print("Loaded:", MODEL_NAME, PRETRAINED)


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Loaded: ViT-B-32 openai


In [4]:
df_all = pd.read_csv(
    ALL_MANIFEST,
    dtype={"pair_id": "string", "image_path": "string", "image_id": "string", "caption": "string"},
    low_memory=False,
)

df_img = df_all[df_all["has_image"] == 1].copy()
if DEBUG_LIMIT_ROWS:
    df_img = df_img.head(DEBUG_LIMIT_ROWS).copy()

df_img["abs_image_path"] = df_img["image_path"].map(lambda p: ROOT / str(p))

broken = df_img[~df_img["abs_image_path"].map(lambda p: p.exists())]
print("All rows:", len(df_all))
print("Rows with local images:", len(df_img))
print("Broken image paths:", len(broken))
if len(broken):
    display(broken.head())


All rows: 97361
Rows with local images: 11621
Broken image paths: 0


In [5]:
ner_all_path = NER_CACHE / "entities_all.pkl"
if not ner_all_path.exists():
    raise FileNotFoundError(f"Missing {ner_all_path}. Run NER notebook first.")

ner_all = pickle.load(open(ner_all_path, "rb"))
print("Loaded NER entries:", len(ner_all))


Loaded NER entries: 97361


In [6]:
@torch.no_grad()
def encode_texts(texts, batch_size=TEXT_BATCH_SIZE):
    if not texts:
        return np.zeros((0, 512), dtype=np.float32)

    vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding text prompts"):
        toks = clip_tokenizer(texts[i:i+batch_size]).to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            v = clip_model.encode_text(toks).float()
        v = v / v.norm(dim=-1, keepdim=True)
        vecs.append(v.cpu().numpy().astype(np.float32))
        del toks, v
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(vecs, axis=0)

@torch.no_grad()
def encode_images(paths, batch_size=IMAGE_BATCH_SIZE):
    if not paths:
        return np.zeros((0, 512), dtype=np.float32)

    vecs = []
    for i in tqdm(range(0, len(paths), batch_size), desc="Encoding images"):
        batch = []
        for p in paths[i:i+batch_size]:
            try:
                img = Image.open(p).convert("RGB")
                batch.append(clip_preprocess(img))
            except Exception:
                batch.append(torch.zeros(3, 224, 224))
        imgs = torch.stack(batch).to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            v = clip_model.encode_image(imgs).float()
        v = v / v.norm(dim=-1, keepdim=True)
        vecs.append(v.cpu().numpy().astype(np.float32))
        del imgs, v, batch
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(vecs, axis=0)


In [7]:
# Build or load image embeddings by unique image_id.
image_emb_path = VG_CACHE / "clip_image_embs_processed.pkl"
REBUILD_IMAGE_EMBS = False

unique_imgs = (
    df_img[["image_id", "abs_image_path"]]
    .drop_duplicates("image_id")
    .reset_index(drop=True)
)

if image_emb_path.exists() and not REBUILD_IMAGE_EMBS:
    image_embs = pickle.load(open(image_emb_path, "rb"))
    print("Loaded image embeddings:", len(image_embs))
else:
    paths = unique_imgs["abs_image_path"].tolist()
    ids = unique_imgs["image_id"].astype(str).tolist()
    arr = encode_images(paths)
    image_embs = {img_id: arr[i] for i, img_id in enumerate(ids)}
    pickle.dump(image_embs, open(image_emb_path, "wb"), protocol=4)
    print("Built image embeddings:", len(image_embs))


Encoding images:   0%|          | 0/719 [00:00<?, ?it/s]

Built image embeddings: 11495


In [8]:
# Entity prompt embeddings.
prompt_emb_path = VG_CACHE / "clip_entity_prompt_embs.pkl"
REBUILD_PROMPT_EMBS = False

prompts = {}
for pid in df_img["pair_id"].astype(str):
    for ent in ner_all.get(pid, []):
        text = str(ent.get("text", "")).strip()
        if text:
            prompts[f"a photo of {text}"] = text

prompt_list = sorted(prompts.keys())
print("Unique entity prompts:", len(prompt_list))

if prompt_emb_path.exists() and not REBUILD_PROMPT_EMBS:
    prompt_embs = pickle.load(open(prompt_emb_path, "rb"))
    print("Loaded prompt embeddings:", len(prompt_embs))
else:
    arr = encode_texts(prompt_list)
    prompt_embs = {prompt_list[i]: arr[i] for i in range(len(prompt_list))}
    pickle.dump(prompt_embs, open(prompt_emb_path, "wb"), protocol=4)
    print("Built prompt embeddings:", len(prompt_embs))


Unique entity prompts: 11478


Encoding text prompts:   0%|          | 0/90 [00:00<?, ?it/s]

Built prompt embeddings: 11478


In [9]:
# Caption CLIP similarity for local-image rows.
caption_sim_path = VG_CACHE / "caption_clip_sim_all.pkl"
REBUILD_CAPTION_SIM = False

if caption_sim_path.exists() and not REBUILD_CAPTION_SIM:
    caption_sim = pickle.load(open(caption_sim_path, "rb"))
    print("Loaded caption similarity:", len(caption_sim))
else:
    caption_sim = {}
    # encode captions in batches
    local_df = df_img[["pair_id", "image_id", "caption"]].copy()
    captions = local_df["caption"].fillna("").astype(str).tolist()
    cap_vecs = encode_texts(captions, batch_size=TEXT_BATCH_SIZE)
    for i, row in local_df.reset_index(drop=True).iterrows():
        img_vec = image_embs.get(str(row["image_id"]))
        if img_vec is None:
            continue
        caption_sim[str(row["pair_id"])] = float(np.dot(img_vec, cap_vecs[i]))
    pickle.dump(caption_sim, open(caption_sim_path, "wb"), protocol=4)
    print("Built caption similarity:", len(caption_sim))


Encoding text prompts:   0%|          | 0/91 [00:00<?, ?it/s]

Built caption similarity: 11621


In [10]:
# Grounding score per entity text.
vg_path = VG_CACHE / "vg_scores_all.pkl"
REBUILD_VG = True

if vg_path.exists() and not REBUILD_VG:
    vg_scores = pickle.load(open(vg_path, "rb"))
    print("Loaded VG scores:", len(vg_scores))
else:
    vg_scores = {}
    rows_iter = df_img[["pair_id", "image_id"]].astype(str).itertuples(index=False)
    for pair_id, image_id in tqdm(list(rows_iter), desc="Scoring entity grounding"):
        img_vec = image_embs.get(image_id)
        if img_vec is None:
            vg_scores[pair_id] = {}
            continue

        scores = {}
        for ent in ner_all.get(pair_id, []):
            text = str(ent.get("text", "")).strip()
            if not text:
                continue
            prompt = f"a photo of {text}"
            txt_vec = prompt_embs.get(prompt)
            if txt_vec is not None:
                scores[text] = float(np.dot(img_vec, txt_vec))
        vg_scores[pair_id] = scores

    # Add empty scores for non-image rows too for consistent lookup.
    for pid in df_all["pair_id"].astype(str):
        vg_scores.setdefault(pid, {})

    pickle.dump(vg_scores, open(vg_path, "wb"), protocol=4)
    print("Built VG scores:", len(vg_scores))


Scoring entity grounding:   0%|          | 0/11621 [00:00<?, ?it/s]

Built VG scores: 97361


In [11]:
# Save dataset-specific VG caches for compatibility.
for dataset_name, sub in df_all.groupby("dataset"):
    d = {str(pid): vg_scores.get(str(pid), {}) for pid in sub["pair_id"].astype(str)}
    pickle.dump(d, open(VG_CACHE / f"vg_{dataset_name}.pkl", "wb"), protocol=4)
    print(dataset_name, len(d), "with scores:", sum(bool(v) for v in d.values()))

# Also split-specific cache.
for (dataset_name, split), sub in df_all.groupby(["dataset", "split"]):
    d = {str(pid): vg_scores.get(str(pid), {}) for pid in sub["pair_id"].astype(str)}
    pickle.dump(d, open(VG_CACHE / f"vg_{dataset_name}_{split}.pkl", "wb"), protocol=4)


mmfakebench 11000 with scores: 7661
newsclippings 85360 with scores: 0
verite 1001 with scores: 584


In [12]:
summary = []
for dataset_name, sub in df_all.groupby("dataset"):
    pids = sub["pair_id"].astype(str).tolist()
    with_scores = sum(bool(vg_scores.get(pid, {})) for pid in pids)
    has_img = int(sub["has_image"].sum())
    summary.append({
        "dataset": dataset_name,
        "rows": len(sub),
        "has_image": has_img,
        "rows_with_vg_scores": with_scores,
    })

vg_summary = pd.DataFrame(summary)
vg_summary.to_csv(TABLES / "visual_grounding_summary.csv", index=False)
vg_summary


,dataset,rows,has_image,rows_with_vg_scores
0,mmfakebench,11000,11000,7661
1,newsclippings,85360,0,0
2,verite,1001,621,584


In [13]:
print("=" * 60)
print("Visual grounding complete.")
print("Cache files:")
for p in sorted(VG_CACHE.glob("*.pkl")):
    print(" ", p.relative_to(ROOT), f"{p.stat().st_size/1e6:.2f} MB")
print("Next: entity_scorer_v2_updated.ipynb")
print("=" * 60)


Visual grounding complete.
Cache files:
  cache\stream_a\visual_grounding\caption_clip_sim_all.pkl 0.39 MB
  cache\stream_a\visual_grounding\clip_entity_prompt_embs.pkl 24.23 MB
  cache\stream_a\visual_grounding\clip_image_embs_processed.pkl 24.72 MB
  cache\stream_a\visual_grounding\vg_mmfake.pkl 0.54 MB
  cache\stream_a\visual_grounding\vg_mmfakebench.pkl 0.75 MB
  cache\stream_a\visual_grounding\vg_mmfakebench_test.pkl 0.68 MB
  cache\stream_a\visual_grounding\vg_mmfakebench_val.pkl 0.07 MB
  cache\stream_a\visual_grounding\vg_newsclippings.pkl 1.39 MB
  cache\stream_a\visual_grounding\vg_newsclippings_test.pkl 0.12 MB
  cache\stream_a\visual_grounding\vg_newsclippings_train.pkl 1.15 MB
  cache\stream_a\visual_grounding\vg_newsclippings_val.pkl 0.11 MB
  cache\stream_a\visual_grounding\vg_scores_all.pkl 2.19 MB
  cache\stream_a\visual_grounding\vg_verite.pkl 0.05 MB
  cache\stream_a\visual_grounding\vg_verite_test.pkl 0.05 MB
Next: entity_scorer_v2_updated.ipynb
